[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/10_pandas_sql/10_5_JOIN.ipynb)

# 10.5: `JOIN`: Combining Tables

Every query so far has worked with one table at a time. But the `measurements` table has no continent column; country-to-continent mapping lives in `countries`. To answer a question like "what is the mean life expectancy per continent in 2007?", you need to bring those two tables together.

In pandas, combining two DataFrames on a shared key is called `merge()`. In SQL, the same operation is called a **`JOIN`**. The logic is identical; only the syntax differs. This notebook covers the two JOIN types you will use in almost every real query: `INNER JOIN` and `LEFT JOIN`.

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_parquet("gapminder.parquet")

countries = df[["country", "continent"]].drop_duplicates().reset_index(drop=True)
measurements = df[["country", "year", "lifeExp", "pop", "gdpPercap"]].copy()

conn = sqlite3.connect(":memory:")
countries.to_sql("countries", conn, index=False, if_exists="replace")
measurements.to_sql("measurements", conn, index=False, if_exists="replace")

print(f"countries: {countries.shape}")
print(f"measurements: {measurements.shape}")

## Why two tables?

The Gapminder data started as one flat file. Splitting it into two tables is artificial here; in the original file, continent is on every row. But this split mimics how data looks in real databases, where redundant information is stored once and referenced by a key rather than repeated in every row.

In a sales database, for example, each order record might contain a `customer_id` but not the customer's name and address; those live in a separate `customers` table and are joined in when needed. The benefit is that if a customer's address changes, it changes in one place, not in thousands of order records. The cost is that answering questions often requires combining tables.

In our case: `measurements` has `country` but not `continent`. `countries` has the mapping. To group measurements by continent, we need to join them.

## The pandas version you already know

Before writing the SQL, here is the pandas `merge()` that produces the combined table:

In [2]:
combined_pandas = measurements.merge(countries, on="country")
print(f"Shape after merge: {combined_pandas.shape}")
combined_pandas.head()

Shape after merge: (1704, 6)


,country,year,lifeExp,pop,gdpPercap,continent
0,Afghanistan,1952,28.801,8425333,779.445314,Asia
1,Afghanistan,1957,30.332,9240934,820.853030,Asia
2,Afghanistan,1962,31.997,10267083,853.100710,Asia
3,Afghanistan,1967,34.020,11537966,836.197138,Asia
4,Afghanistan,1972,36.088,13079460,739.981106,Asia


1,704 rows, matching every row from `measurements` to its corresponding row in `countries` on the shared `country` column. The result has all the original columns from both tables. This is what a JOIN does: it finds rows in two tables that share the same value in a specified column and combines them into a single wider row.

## `INNER JOIN`: the SQL equivalent of `merge(how="inner")`

The SQL syntax for a JOIN puts the two tables in `FROM` and `JOIN`, connected by an `ON` clause that names the shared key. Table aliases (`AS m`, `AS c`) shorten the table names to a single letter so column references (`m.country`, `c.continent`) stay readable.

In [3]:
pd.read_sql("""
    SELECT m.country,
           c.continent,
           m.year,
           m.lifeExp,
           m.gdpPercap
    FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    LIMIT 8
""", conn)

,country,continent,year,lifeExp,gdpPercap
0,Afghanistan,Asia,1952,28.801,779.445314
1,Afghanistan,Asia,1957,30.332,820.853030
2,Afghanistan,Asia,1962,31.997,853.100710
3,Afghanistan,Asia,1967,34.020,836.197138
4,Afghanistan,Asia,1972,36.088,739.981106
5,Afghanistan,Asia,1977,38.438,786.113360
6,Afghanistan,Asia,1982,39.854,978.011439
7,Afghanistan,Asia,1987,40.822,852.395945


The result looks exactly like the pandas merge output: `continent` from `countries` has been brought in alongside the measurement columns. The `ON m.country = c.country` clause is the join condition; it specifies which column in each table holds the matching key.

The alias syntax (`AS m`, `AS c`) is optional but strongly recommended whenever you reference columns from both tables. Without it, `country` would be ambiguous, since both tables have that column, and the database would raise an error.

| pandas | SQL |
|---|---|
| `measurements.merge(countries, on="country")` | `FROM measurements AS m INNER JOIN countries AS c ON m.country = c.country` |

## What `INNER JOIN` drops

An INNER JOIN keeps only rows where the key exists in both tables. Any row in either table with no matching row in the other is silently dropped. To make this concrete, let's add a country to `measurements` that does not exist in `countries`.

In [4]:
# Add a fake country to measurements
fake_row = pd.DataFrame([{"country": "Narnia", "year": 2007,
                           "lifeExp": 999.0, "pop": 1000, "gdpPercap": 50000.0}])
measurements_with_fake = pd.concat([measurements, fake_row], ignore_index=True)
measurements_with_fake.to_sql("measurements", conn, index=False, if_exists="replace")

# INNER JOIN: Narnia disappears because it has no match in countries
result_inner = pd.read_sql("""
    SELECT m.country, c.continent, m.year, m.lifeExp
    FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    WHERE m.country IN ('Afghanistan', 'Narnia')
""", conn)
print("INNER JOIN result:")
result_inner

INNER JOIN result:


,country,continent,year,lifeExp
0,Afghanistan,Asia,1952,28.801
1,Afghanistan,Asia,1957,30.332
2,Afghanistan,Asia,1962,31.997
3,Afghanistan,Asia,1967,34.020
4,Afghanistan,Asia,1972,36.088
5,Afghanistan,Asia,1977,38.438
6,Afghanistan,Asia,1982,39.854
7,Afghanistan,Asia,1987,40.822
8,Afghanistan,Asia,1992,41.674
9,Afghanistan,Asia,1997,41.763


Narnia's row is gone. The INNER JOIN found no matching row in `countries` for `country = 'Narnia'`, so the entire row was excluded from the result. Afghanistan's 12 rows are all present because Afghanistan appears in both tables.

This behavior is usually what you want, since you are asking for data that exists in both tables. But sometimes a missing match is meaningful information: a measurement with no country record, or an order with no customer. For those cases, you need `LEFT JOIN`.

## `LEFT JOIN`: keeping all rows from the left table

A `LEFT JOIN` keeps every row from the left table (the one in `FROM`), even if there is no matching row in the right table. Where there is no match, the columns from the right table are filled with `NULL`. This is the SQL equivalent of `merge(how="left")`.

In [5]:
# LEFT JOIN: Narnia stays, but its continent is NULL
result_left = pd.read_sql("""
    SELECT m.country, c.continent, m.year, m.lifeExp
    FROM measurements AS m
    LEFT JOIN countries AS c ON m.country = c.country
    WHERE m.country IN ('Afghanistan', 'Narnia')
""", conn)
print("LEFT JOIN result:")
result_left

LEFT JOIN result:


,country,continent,year,lifeExp
0,Afghanistan,Asia,1952,28.801
1,Afghanistan,Asia,1957,30.332
2,Afghanistan,Asia,1962,31.997
3,Afghanistan,Asia,1967,34.020
4,Afghanistan,Asia,1972,36.088
5,Afghanistan,Asia,1977,38.438
6,Afghanistan,Asia,1982,39.854
7,Afghanistan,Asia,1987,40.822
8,Afghanistan,Asia,1992,41.674
9,Afghanistan,Asia,1997,41.763


Now Narnia appears in the result, but its `continent` column is `None` (SQL `NULL`) because the `countries` table has no row for it. The left table (`measurements`) was preserved in full; the right table (`countries`) contributed what it could.

This is how you use LEFT JOIN to find unmatched rows: filter the result for `WHERE c.continent IS NULL` and you get only the rows in `measurements` that had no match in `countries`.

| pandas | SQL |
|---|---|
| `measurements.merge(countries, on="country", how="inner")` | `INNER JOIN` |
| `measurements.merge(countries, on="country", how="left")` | `LEFT JOIN` |

The Narnia row added above is still in the database. The cell below overwrites the `measurements` table with the original data so the remaining examples work with real Gapminder observations.

In [6]:
# Restore the real measurements table (without Narnia)
measurements.to_sql("measurements", conn, index=False, if_exists="replace")
print("Measurements restored.")

Measurements restored.


## JOIN + WHERE + GROUP BY together

In notebooks 10.3 and 10.4, several queries already used a JOIN to bring in the continent column before filtering or grouping. Now that the JOIN mechanics are explicit, here is a full example that chains all four clauses: JOIN to combine tables, WHERE to filter rows, GROUP BY to aggregate, HAVING to filter groups.

In [7]:
pd.read_sql("""
    SELECT c.continent,
           COUNT(DISTINCT m.country) AS n_countries,
           ROUND(AVG(m.lifeExp), 1) AS avg_life,
           ROUND(AVG(m.gdpPercap), 0) AS avg_gdp
    FROM measurements AS m
    INNER JOIN countries AS c ON m.country = c.country
    WHERE m.year = 2007
    GROUP BY c.continent
    HAVING COUNT(DISTINCT m.country) >= 10
    ORDER BY avg_life DESC
""", conn)

,continent,n_countries,avg_life,avg_gdp
0,Europe,30,77.6,25054.0
1,Americas,25,73.6,11003.0
2,Asia,33,70.7,12473.0
3,Africa,52,54.8,3089.0


Oceania (2 countries) is excluded by the `HAVING COUNT(DISTINCT country) >= 10` condition. The remaining four continents each have at least 25 countries. The query reads like a plain-English description of the analysis: combine the tables, keep only 2007 data, compute per-continent summaries, drop continents with fewer than 10 countries, sort by life expectancy.

## A note on JOIN types

SQL has four JOIN types: `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, and `FULL OUTER JOIN`. In practice, `RIGHT JOIN` and `FULL OUTER JOIN` are rarely used in analytical queries because you can always rewrite a `RIGHT JOIN` as a `LEFT JOIN` with the tables swapped. Learning `INNER` and `LEFT` covers the vast majority of real-world cases.

SQLite does not support `RIGHT JOIN` or `FULL OUTER JOIN`, another reason to not worry about them here.

The complete clause order for a SQL query is now:

```
SELECT   ...
FROM     ...
JOIN     ... ON ...
WHERE    ...
GROUP BY ...
HAVING   ...
ORDER BY ...
LIMIT    ...
```

Not every query uses every clause, but when multiple clauses appear, they must come in this order.

## Updated translation table

| Goal | pandas | SQL |
|---|---|---|
| Select columns | `df[["a", "b"]]` | `SELECT a, b FROM table` |
| Filter rows | `df.query("col > x")` | `WHERE col > x` |
| Sort | `df.sort_values("col")` | `ORDER BY col` |
| First N rows | `df.head(N)` | `LIMIT N` |
| Count rows per group | `groupby(col).size()` | `COUNT(*) ... GROUP BY col` |
| Mean per group | `groupby(col)["val"].mean()` | `AVG(val) ... GROUP BY col` |
| Count distinct | `.nunique()` | `COUNT(DISTINCT col)` |
| Filter groups | `groupby(col).filter(lambda g: ...)` | `HAVING condition` |
| Combine tables (inner) | `df1.merge(df2, on="key")` | `FROM t1 INNER JOIN t2 ON t1.key = t2.key` |
| Combine tables (left) | `df1.merge(df2, on="key", how="left")` | `FROM t1 LEFT JOIN t2 ON t1.key = t2.key` |

## What's next

You now have the full SQL vocabulary for analytical queries: SELECT, WHERE, GROUP BY, HAVING, and JOIN. Notebook 10.6 introduces one more concept, subqueries, where one SELECT is nested inside another, and then steps back to compare SQL and pandas as tools: what each does best, and how to choose between them for a given problem.